In [ ]:
!pip install tokenizers

# Phase 2 & 3: Baseline BPE + SuperBPE Tokenizer Training

TRI-Project. Trains a standard BPE baseline and a two-stage SuperBPE
tokenizer on a temperature-sampled (α=0.3) combined corpus across 9
African languages, then measures fragmentation on held-out eval data.

Phase 1 (data engineering) lives in `scripts/` in the repo root.
See README.md for full methodology and results.

Phase 2


In [ ]:
path = "/content/drive/MyDrive/TRI-Project/am_wikipedia/cleaned/corpus.txt"

with open(path, "r", encoding="utf-8") as f:
    lines = f.readlines()

word_count = sum(len(line.split()) for line in lines)

print(f"Number of lines: {len(lines):,}")
print(f"Number of words: {word_count:,}")
print("First line preview:", lines[0][:200])

Number of lines: 8,522
Number of words: 1,292,244
First line preview: ኤድዊን አቦት አቦት FBA (ታህሳስ 20 1838 - ጥቅምት 12 1926) የእንግሊዛዊ መምህር እና የሃይማኖት ምሁር ነበር፣ በይበልጥ የኖቬላ "ፍላት ላንድ" (1884) ደራሲ በመባል ይታወቃል። ኤድዊን አቦት አቦት፤ የኤድዊን አቦት (1808 – 1882) በሜሪሌቦን የፊሎሎጂ ትምህርት ቤት ርእሰ መምህር እና ሚስቱ ጄ


In [ ]:
import os

base = "/content/drive/MyDrive/TRI-Project"

corpus_paths = {
    "am": f"{base}/am_wikipedia/cleaned/corpus.txt",
    "ha": f"{base}/ha_wikipedia/cleaned/corpus.txt",
    "ig": f"{base}/ig_wikipedia/cleaned/corpus.txt",
    "ny": f"{base}/ny_wikipedia/cleaned/corpus.txt",
    "rw": f"{base}/rw_wikipedia/cleaned/corpus.txt",
    "sw": f"{base}/sw_wikipedia/cleaned/corpus.txt",
    "wo": f"{base}/wo_combined/cleaned/corpus.txt",   # Wolof uses the combined folder, not wo_wikipedia
    "yo": f"{base}/yo_wikipedia/cleaned/corpus.txt",
    "zu": f"{base}/zu_wikipedia/cleaned/corpus.txt",
}

for lang, path in corpus_paths.items():
    exists = os.path.exists(path)
    print(f"{lang}: {'FOUND' if exists else 'MISSING'} -> {path}")

am: FOUND -> /content/drive/MyDrive/TRI-Project/am_wikipedia/cleaned/corpus.txt
ha: FOUND -> /content/drive/MyDrive/TRI-Project/ha_wikipedia/cleaned/corpus.txt
ig: FOUND -> /content/drive/MyDrive/TRI-Project/ig_wikipedia/cleaned/corpus.txt
ny: FOUND -> /content/drive/MyDrive/TRI-Project/ny_wikipedia/cleaned/corpus.txt
rw: FOUND -> /content/drive/MyDrive/TRI-Project/rw_wikipedia/cleaned/corpus.txt
sw: FOUND -> /content/drive/MyDrive/TRI-Project/sw_wikipedia/cleaned/corpus.txt
wo: FOUND -> /content/drive/MyDrive/TRI-Project/wo_combined/cleaned/corpus.txt
yo: FOUND -> /content/drive/MyDrive/TRI-Project/yo_wikipedia/cleaned/corpus.txt
zu: FOUND -> /content/drive/MyDrive/TRI-Project/zu_wikipedia/cleaned/corpus.txt


In [ ]:
word_counts = {}

for lang, path in corpus_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    word_counts[lang] = sum(len(line.split()) for line in lines)
    print(f"{lang}: {len(lines):,} lines, {word_counts[lang]:,} words")

am: 8,522 lines, 1,292,244 words
ha: 104,631 lines, 46,045,081 words
ig: 53,686 lines, 19,886,841 words
ny: 892 lines, 204,640 words
rw: 11,125 lines, 1,931,619 words
sw: 86,866 lines, 10,258,695 words
wo: 3,721 lines, 458,166 words
yo: 13,833 lines, 2,990,168 words
zu: 3,859 lines, 695,910 words


In [ ]:
alpha = 0.3
weighted = {lang: n ** alpha for lang, n in word_counts.items()}
total_weighted = sum(weighted.values())
target_share = {lang: w / total_weighted for lang, w in weighted.items()}

total_budget = sum(word_counts.values())  # keep overall corpus size the same
target_words = {lang: int(target_share[lang] * total_budget) for lang in word_counts}

for lang in word_counts:
    print(f"{lang}: raw={word_counts[lang]:,}  target={target_words[lang]:,}")

am: raw=1,292,244  target=6,644,461
ha: raw=46,045,081  target=19,409,369
ig: raw=19,886,841  target=15,087,809
ny: raw=204,640  target=3,822,553
rw: raw=1,931,619  target=7,496,057
sw: raw=10,258,695  target=12,370,409
wo: raw=458,166  target=4,868,142
yo: raw=2,990,168  target=8,546,043
zu: raw=695,910  target=5,518,516


In [ ]:
import random

rng = random.Random(42)  # fixed seed, same one you used for eval splits

def build_sample(lines, target_words, rng):
    rng.shuffle(lines)
    selected = []
    words_so_far = 0
    idx = 0
    n = len(lines)
    while words_so_far < target_words:
        if idx == n:              # used every line once -> reshuffle, start again
            rng.shuffle(lines)
            idx = 0
        line = lines[idx]
        selected.append(line)
        words_so_far += len(line.split())
        idx += 1
    return selected

combined_lines = []

for lang, path in corpus_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        lines = [l.rstrip("\n") for l in f if l.strip()]
    sampled = build_sample(lines, target_words[lang], rng)
    combined_lines.extend(sampled)
    print(f"{lang}: sampled {len(sampled):,} lines (~{target_words[lang]:,} words)")

print(f"\nTotal combined lines: {len(combined_lines):,}")

am: sampled 43,817 lines (~6,644,461 words)
ha: sampled 43,874 lines (~19,409,369 words)
ig: sampled 40,835 lines (~15,087,809 words)
ny: sampled 16,680 lines (~3,822,553 words)
rw: sampled 43,117 lines (~7,496,057 words)
sw: sampled 104,660 lines (~12,370,409 words)
wo: sampled 39,329 lines (~4,868,142 words)
yo: sampled 39,587 lines (~8,546,043 words)
zu: sampled 30,579 lines (~5,518,516 words)

Total combined lines: 402,478


In [ ]:
rng.shuffle(combined_lines)

output_path = "/content/drive/MyDrive/TRI-Project/combined_corpus.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for line in combined_lines:
        f.write(line + "\n")

print(f"Saved {len(combined_lines):,} lines to {output_path}")

Saved 402,478 lines to /content/drive/MyDrive/TRI-Project/combined_corpus.txt


In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

trainer = BpeTrainer(
    vocab_size=24000,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)

In [ ]:
tokenizer.train(files=["/content/drive/MyDrive/TRI-Project/combined_corpus.txt"], trainer=trainer)

In [ ]:
save_path = "/content/drive/MyDrive/TRI-Project/baseline_bpe_tokenizer.json"
tokenizer.save(save_path)
print(f"Saved to {save_path}")

# quick sanity check on one Amharic sentence and one Hausa sentence
test_sentences = {
    "am": "ኤድዊን አቦት አቦት FBA",
    "ha": "Wannan yare ne na Afirka",
}

for lang, text in test_sentences.items():
    encoding = tokenizer.encode(text)
    print(f"\n{lang}: '{text}'")
    print("tokens:", encoding.tokens)
    print("token count:", len(encoding.tokens))

Saved to /content/drive/MyDrive/TRI-Project/baseline_bpe_tokenizer.json

am: 'ኤድዊን አቦት አቦት FBA'
tokens: ['áĬ¤', 'áĭµ', 'áĭĬ', 'áĬķ', 'ĠáĬł', 'áī¦áīµ', 'ĠáĬł', 'áī¦áīµ', 'ĠF', 'BA']
token count: 10

ha: 'Wannan yare ne na Afirka'
tokens: ['W', 'annan', 'Ġyare', 'Ġne', 'Ġna', 'ĠAfirka']
token count: 6


In [ ]:
for lang, text in test_sentences.items():
    encoding = tokenizer.encode(text)
    decoded = tokenizer.decode(encoding.ids)
    print(f"{lang} original: {text}")
    print(f"{lang} decoded:  {decoded}")
    print(f"match: {decoded.strip() == text.strip()}\n")

am original: ኤድዊን አቦት አቦት FBA
am decoded:  áĬ¤ áĭµ áĭĬ áĬķ ĠáĬł áī¦áīµ ĠáĬł áī¦áīµ ĠF BA
match: False

ha original: Wannan yare ne na Afirka
ha decoded:  W annan Ġyare Ġne Ġna ĠAfirka
match: False



In [ ]:
from tokenizers import decoders

tokenizer.decoder = decoders.ByteLevel()

# re-run the same check
for lang, text in test_sentences.items():
    encoding = tokenizer.encode(text)
    decoded = tokenizer.decode(encoding.ids)
    print(f"{lang} original: {text}")
    print(f"{lang} decoded:  {decoded}")
    print(f"match: {decoded.strip() == text.strip()}\n")

am original: ኤድዊን አቦት አቦት FBA
am decoded:  ኤድዊን አቦት አቦት FBA
match: True

ha original: Wannan yare ne na Afirka
ha decoded:  Wannan yare ne na Afirka
match: True



In [ ]:
tokenizer.save(save_path)
print(f"Re-saved with correct decoder to {save_path}")

Re-saved with correct decoder to /content/drive/MyDrive/TRI-Project/baseline_bpe_tokenizer.json


In [ ]:
eval_paths = {
    "am": f"{base}/eval_holdout/am_wikipedia/eval.txt",
    "ha": f"{base}/eval_holdout/ha_wikipedia/eval.txt",
    "ig": f"{base}/eval_holdout/ig_wikipedia/eval.txt",
    "ny": f"{base}/eval_holdout/ny_wikipedia/eval.txt",
    "rw": f"{base}/eval_holdout/rw_wikipedia/eval.txt",
    "sw": f"{base}/eval_holdout/sw_wikipedia/eval.txt",
    "wo": f"{base}/eval_holdout/wo_wikipedia/eval.txt",   # double-check this one
    "yo": f"{base}/eval_holdout/yo_wikipedia/eval.txt",
    "zu": f"{base}/eval_holdout/zu_wikipedia/eval.txt",
}

for lang, path in eval_paths.items():
    print(f"{lang}: {'FOUND' if os.path.exists(path) else 'MISSING'} -> {path}")

am: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/am_wikipedia/eval.txt
ha: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/ha_wikipedia/eval.txt
ig: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/ig_wikipedia/eval.txt
ny: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/ny_wikipedia/eval.txt
rw: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/rw_wikipedia/eval.txt
sw: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/sw_wikipedia/eval.txt
wo: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/wo_wikipedia/eval.txt
yo: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/yo_wikipedia/eval.txt
zu: FOUND -> /content/drive/MyDrive/TRI-Project/eval_holdout/zu_wikipedia/eval.txt


In [ ]:
fragmentation_results = {}

for lang, path in eval_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        lines = [l.rstrip("\n") for l in f if l.strip()]

    total_words = 0
    total_tokens = 0

    for line in lines:
        total_words += len(line.split())
        total_tokens += len(tokenizer.encode(line).tokens)

    ratio = total_tokens / total_words
    fragmentation_results[lang] = ratio
    print(f"{lang}: {total_words:,} words -> {total_tokens:,} tokens  (ratio: {ratio:.3f} tokens/word)")

am: 65,247 words -> 189,051 tokens  (ratio: 2.897 tokens/word)
ha: 2,270,189 words -> 3,255,483 tokens  (ratio: 1.434 tokens/word)
ig: 984,426 words -> 1,522,266 tokens  (ratio: 1.546 tokens/word)
ny: 8,046 words -> 14,236 tokens  (ratio: 1.769 tokens/word)
rw: 98,395 words -> 190,274 tokens  (ratio: 1.934 tokens/word)
sw: 505,402 words -> 796,951 tokens  (ratio: 1.577 tokens/word)
wo: 29,236 words -> 48,425 tokens  (ratio: 1.656 tokens/word)
yo: 142,692 words -> 259,077 tokens  (ratio: 1.816 tokens/word)
zu: 40,752 words -> 101,443 tokens  (ratio: 2.489 tokens/word)


In [ ]:
import json

with open("/content/drive/MyDrive/TRI-Project/baseline_fragmentation_results.json", "w") as f:
    json.dump(fragmentation_results, f, indent=2)

print("Saved baseline fragmentation results.")

Saved baseline fragmentation results.


Phase 3


In [ ]:
from tokenizers import Tokenizer, decoders
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

stage1_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
stage1_tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
stage1_tokenizer.decoder = decoders.ByteLevel()

stage1_trainer = BpeTrainer(
    vocab_size=19200,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)

stage1_tokenizer.train(
    files=["/content/drive/MyDrive/TRI-Project/combined_corpus.txt"],
    trainer=stage1_trainer,
)

stage1_tokenizer.save("/content/drive/MyDrive/TRI-Project/superbpe_stage1_tokenizer.json")
print("Stage 1 done. Vocab size:", stage1_tokenizer.get_vocab_size())

Stage 1 done. Vocab size: 19200


In [ ]:
# map each of the 19,200 stage-1 tokens to one unique placeholder character
vocab = stage1_tokenizer.get_vocab()  # {token_string: token_id}
id_to_placeholder = {tok_id: chr(0xF0000 + tok_id) for tok_id in vocab.values()}

input_path = "/content/drive/MyDrive/TRI-Project/combined_corpus.txt"
meta_corpus_path = "/content/drive/MyDrive/TRI-Project/stage2_meta_corpus.txt"

with open(input_path, "r", encoding="utf-8") as fin, \
     open(meta_corpus_path, "w", encoding="utf-8") as fout:
    for line in fin:
        line = line.rstrip("\n")
        if not line.strip():
            continue
        ids = stage1_tokenizer.encode(line).ids
        placeholder_line = "".join(id_to_placeholder[i] for i in ids)
        fout.write(placeholder_line + "\n")

print("Meta-corpus written.")

Meta-corpus written.


In [ ]:
stage2_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
# NOTE: no pre_tokenizer set here on purpose — that's what lifts the whitespace restriction

stage2_trainer = BpeTrainer(
    vocab_size=24000,   # 19,200 (stage 1 alphabet) + ~4,800 new superword merges
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    show_progress=True,
)

stage2_tokenizer.train(
    files=["/content/drive/MyDrive/TRI-Project/stage2_meta_corpus.txt"],
    trainer=stage2_trainer,
)

stage2_tokenizer.save("/content/drive/MyDrive/TRI-Project/superbpe_stage2_raw.json")
print("Stage 2 done. Vocab size:", stage2_tokenizer.get_vocab_size())

Stage 2 done. Vocab size: 24000


In [ ]:
# reverse map: placeholder character -> stage-1 token id
placeholder_to_id = {v: k for k, v in id_to_placeholder.items()}
stage1_id_to_token = {v: k for k, v in stage1_tokenizer.get_vocab().items()}

stage2_vocab = stage2_tokenizer.get_vocab()  # {placeholder-string: stage2_id}

# for every stage-2 token, reconstruct the real (human-readable) string it stands for
stage2_id_to_realstring = {}
for placeholder_str, stage2_id in stage2_vocab.items():
    stage1_ids = [placeholder_to_id[ch] for ch in placeholder_str if ch in placeholder_to_id]
    real_string = "".join(stage1_id_to_token.get(i, "") for i in stage1_ids)
    stage2_id_to_realstring[stage2_id] = real_string

def superbpe_encode(text):
    """Full pipeline: text -> stage1 ids -> placeholder string -> stage2 ids"""
    stage1_ids = stage1_tokenizer.encode(text).ids
    placeholder_str = "".join(id_to_placeholder[i] for i in stage1_ids)
    stage2_ids = stage2_tokenizer.encode(placeholder_str).ids
    return stage2_ids

def superbpe_tokens_readable(text):
    """Same as above, but returns human-readable token strings instead of ids"""
    ids = superbpe_encode(text)
    return [stage2_id_to_realstring[i] for i in ids]

# quick check on the same two test sentences from Phase 2
for lang, text in test_sentences.items():
    tokens = superbpe_tokens_readable(text)
    print(f"{lang}: '{text}'")
    print("superbpe tokens:", tokens)
    print("token count:", len(tokens), "\n")

am: 'ኤድዊን አቦት አቦት FBA'
superbpe tokens: ['áĬ¤', 'áĭµ', 'áĭĬ', 'áĬķĠáĬł', 'áī¦áīµ', 'ĠáĬł', 'áī¦áīµ', 'ĠF', 'BA']
token count: 9 

ha: 'Wannan yare ne na Afirka'
superbpe tokens: ['W', 'annan', 'Ġyare', 'ĠneĠna', 'ĠAfirka']
token count: 5 



In [ ]:
superbpe_fragmentation_results = {}

for lang, path in eval_paths.items():
    with open(path, "r", encoding="utf-8") as f:
        lines = [l.rstrip("\n") for l in f if l.strip()]

    total_words = 0
    total_tokens = 0

    for line in lines:
        total_words += len(line.split())
        total_tokens += len(superbpe_encode(line))

    ratio = total_tokens / total_words
    superbpe_fragmentation_results[lang] = ratio
    print(f"{lang}: {total_words:,} words -> {total_tokens:,} tokens  (ratio: {ratio:.3f} tokens/word)")

am: 65,247 words -> 184,670 tokens  (ratio: 2.830 tokens/word)
ha: 2,270,189 words -> 2,677,238 tokens  (ratio: 1.179 tokens/word)
ig: 984,426 words -> 1,203,414 tokens  (ratio: 1.222 tokens/word)
ny: 8,046 words -> 13,536 tokens  (ratio: 1.682 tokens/word)
rw: 98,395 words -> 174,237 tokens  (ratio: 1.771 tokens/word)
sw: 505,402 words -> 706,377 tokens  (ratio: 1.398 tokens/word)
wo: 29,236 words -> 46,208 tokens  (ratio: 1.581 tokens/word)
yo: 142,692 words -> 214,330 tokens  (ratio: 1.502 tokens/word)
zu: 40,752 words -> 98,104 tokens  (ratio: 2.407 tokens/word)


In [ ]:
import json
with open("/content/drive/MyDrive/TRI-Project/superbpe_fragmentation_results.json", "w") as f:
    json.dump(superbpe_fragmentation_results, f, indent=2)
print("Saved.")

Saved.


In [ ]:
Phase 4


In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

try:
    import transformers
    print("transformers version:", transformers.__version__)
except ImportError:
    print("transformers NOT installed yet")

GPU available: True
GPU name: Tesla T4
transformers version: 5.13.1


In [ ]:
with open("/content/drive/MyDrive/TRI-Project/combined_corpus.txt", "r", encoding="utf-8") as f:
    all_lines = [l.rstrip("\n") for l in f if l.strip()]

subset_size = 20000
eval_size = 1000

subset_lines = all_lines[:subset_size]
train_lines = subset_lines[:-eval_size]
eval_lines = subset_lines[-eval_size:]

train_path = "/content/drive/MyDrive/TRI-Project/model_train_text.txt"
eval_path = "/content/drive/MyDrive/TRI-Project/model_eval_text.txt"

with open(train_path, "w", encoding="utf-8") as f:
    for line in train_lines:
        f.write(line + "\n")

with open(eval_path, "w", encoding="utf-8") as f:
    for line in eval_lines:
        f.write(line + "\n")

print(f"Train lines: {len(train_lines):,}")
print(f"Eval lines: {len(eval_lines):,}")

# byte counts -- we'll need these later for the fair bits-per-byte comparison
train_bytes = sum(len(line.encode("utf-8")) for line in train_lines)
eval_bytes = sum(len(line.encode("utf-8")) for line in eval_lines)
print(f"Train bytes: {train_bytes:,}")
print(f"Eval bytes: {eval_bytes:,}")

Train lines: 19,000
Eval lines: 1,000
Train bytes: 27,930,973
Eval bytes: 1,415,791


In [ ]:
try:
    print("Baseline tokenizer:", tokenizer.get_vocab_size())
    print("Stage 1:", stage1_tokenizer.get_vocab_size())
    print("Stage 2:", stage2_tokenizer.get_vocab_size())
    print("superbpe_encode function exists:", callable(superbpe_encode))
    print("All good, everything is still loaded.")
except NameError as e:
    print("Missing:", e)
    print("We'll need to reload from saved files.")

Missing: name 'tokenizer' is not defined
We'll need to reload from saved files.


In [ ]:
from tokenizers import Tokenizer, decoders
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

base = "/content/drive/MyDrive/TRI-Project"

# baseline tokenizer (Phase 2)
tokenizer = Tokenizer.from_file(f"{base}/baseline_bpe_tokenizer.json")

# stage 1 and stage 2 (Phase 3)
stage1_tokenizer = Tokenizer.from_file(f"{base}/superbpe_stage1_tokenizer.json")
stage2_tokenizer = Tokenizer.from_file(f"{base}/superbpe_stage2_raw.json")

print("Baseline vocab:", tokenizer.get_vocab_size())
print("Stage 1 vocab:", stage1_tokenizer.get_vocab_size())
print("Stage 2 vocab:", stage2_tokenizer.get_vocab_size())

Baseline vocab: 24000
Stage 1 vocab: 19200
Stage 2 vocab: 24000


In [ ]:
vocab = stage1_tokenizer.get_vocab()
id_to_placeholder = {tok_id: chr(0xF0000 + tok_id) for tok_id in vocab.values()}
placeholder_to_id = {v: k for k, v in id_to_placeholder.items()}
stage1_id_to_token = {v: k for k, v in stage1_tokenizer.get_vocab().items()}

stage2_vocab = stage2_tokenizer.get_vocab()
stage2_id_to_realstring = {}
for placeholder_str, stage2_id in stage2_vocab.items():
    stage1_ids = [placeholder_to_id[ch] for ch in placeholder_str if ch in placeholder_to_id]
    real_string = "".join(stage1_id_to_token.get(i, "") for i in stage1_ids)
    stage2_id_to_realstring[stage2_id] = real_string

def superbpe_encode(text):
    stage1_ids = stage1_tokenizer.encode(text).ids
    placeholder_str = "".join(id_to_placeholder[i] for i in stage1_ids)
    stage2_ids = stage2_tokenizer.encode(placeholder_str).ids
    return stage2_ids

def superbpe_tokens_readable(text):
    ids = superbpe_encode(text)
    return [stage2_id_to_realstring[i] for i in ids]

# quick sanity check
test_text = "Wannan yare ne na Afirka"
print("Baseline tokens:", len(tokenizer.encode(test_text).ids))
print("SuperBPE tokens:", len(superbpe_encode(test_text)))

Baseline tokens: 6
SuperBPE tokens: 5


In [ ]:
def tokenize_file_baseline(path):
    all_ids = []
    eos_id = tokenizer.token_to_id("[EOS]")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip():
                all_ids.extend(tokenizer.encode(line).ids)
                all_ids.append(eos_id)
    return all_ids

def tokenize_file_superbpe(path):
    all_ids = []
    eos_id = stage2_tokenizer.token_to_id("[EOS]")
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.strip():
                all_ids.extend(superbpe_encode(line))
                all_ids.append(eos_id)
    return all_ids

train_ids_baseline = tokenize_file_baseline(train_path)
eval_ids_baseline = tokenize_file_baseline(eval_path)

train_ids_superbpe = tokenize_file_superbpe(train_path)
eval_ids_superbpe = tokenize_file_superbpe(eval_path)

print(f"Baseline: {len(train_ids_baseline):,} train tokens, {len(eval_ids_baseline):,} eval tokens")
print(f"SuperBPE: {len(train_ids_superbpe):,} train tokens, {len(eval_ids_superbpe):,} eval tokens")

Baseline: 6,885,275 train tokens, 353,529 eval tokens
SuperBPE: 6,070,085 train tokens, 310,326 eval tokens
